In [1]:
# Library imports
import torch
import math
import random
import json
import tqdm
import os
import spacy
import re

import numpy as np

from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer

from sklearn.metrics.pairwise import cosine_similarity

from dotenv import dotenv_values

from src.claim_extraction.config import DOTENV_PATH
from src.claim_extraction.extractor import extract_claims

In [2]:
# Load env settings

env = dotenv_values(DOTENV_PATH)

In [3]:
# Determine what device to use
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

if device.type == 'cuda':
    device_name = torch.cuda.get_device_name(device)

    print(f'CUDA on {device_name}')
else:
    print(f'CPU')

CUDA on NVIDIA GeForce RTX 3060 Laptop GPU


In [4]:
# Check if everything's cleaned
allocated = torch.cuda.memory_allocated() / (1024 * 1024)
reserved = torch.cuda.memory_reserved() / (1024 * 1024)

print(f'Allocated: {allocated:.1f} MB; Reserved: {reserved:.1f} MB')

Allocated: 0.0 MB; Reserved: 0.0 MB


In [5]:
# Configuration

DETECT_MODE = 'claims'
CLAIMS_INPUT_DIR = './extracted_claims/pass_01_initial_claim_extraction'

NLI_MODEL = 'MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli' # https://huggingface.co/MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli
EMBED_MODEL = 'sentence-transformers/all-MiniLM-L6-v2' # https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2

CLAIM_PREFIXES = [ 'In the beginning', 'Later', 'In the end']

# Standard generation parameters used by all runs
CLAIM_MODEL_NAME = env.get('CLAIM_MODEL_1', 'Qwen/Qwen3.5-4B')
CLAIM_MODEL_TEMPERATURE = 0.1
CLAIM_MODEL_MAX_NEW_TOKENS = 4096
CLAIM_MODEL_BACKEND = 'local'
CLAIM_MODEL_VERBOSE = True

In [6]:
# Create models

nli_tokenizer = AutoTokenizer.from_pretrained(NLI_MODEL)
nli_model = AutoModelForSequenceClassification.from_pretrained(NLI_MODEL).to(device)

embed_model = SentenceTransformer(EMBED_MODEL).to(device)

nlp = spacy.load("en_core_web_sm")

Loading weights:   0%|          | 0/394 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [7]:
# Load dataset

dataset = json.load(open('datasets/ContraDoc/ContraDoc.json', 'r'))

# Label examples
for example in dataset['pos'].values():
	example['label'] = 1.0 

for example in dataset['neg'].values():
	example['label'] = 0.0 

# Flatten list
all_examples = { **dataset['pos'], **dataset['neg'] }

print(f'Dataset loaded with {len(dataset["pos"])} positive and {len(dataset["neg"])} negative examples.')

Dataset loaded with 449 positive and 442 negative examples.


In [22]:
# Model functions

def normalize_punctuation(text: str) -> str:
    # remove spaces before punctuation
    text = re.sub(r"\s+([,.;:!?])", r"\1", text)

    # ensure space after punctuation if followed by word
    text = re.sub(r"([,.;:!?])([A-Za-z])", r"\1 \2", text)

    # normalize quotes spacing
    text = re.sub(r"\s+([\"'])", r" \1", text)
    text = re.sub(r"([\"'])\s+", r"\1 ", text)

    # normalize dash spacing
    text = text.replace(" - ", "-")

    # collapse whitespace
    #text = re.sub(r"\s+", " ", text)

    return text.strip()

def remove_structural_lines(text):
    lines = text.split("\n")
    clean = []
    
    for line in lines:
        l = line.strip()

        if not l:
            continue

        # wikipedia headers
        if re.match(r"^\s*(=+\s*)+[^=]+?(\s*=)+\s*$", l):
            continue

        # markdown headers
        if re.match(r"^#+\s", l):
            continue

        # numbered lists
        if re.match(r"^\d+\s*[.)]\s*", l):
            continue

        # bullet lists
        if re.match(r"^[-*]\s+", l):
            continue

        # table rows
        if "|" in l and l.count("|") >= 2:
            continue

        clean.append(l)

    return " ".join(clean)

def normalize_numbers(text: str) -> str:
    # Weird commas
    text = text.replace("@,@", ",")

    # decimal numbers: "2 . 9" -> "2.9"
    text = re.sub(r"(\d)\s*\.\s*(\d)", r"\1.\2", text)

    # thousands separators: "1 , 000" -> "1,000"
    text = re.sub(r"(\d)\s*,\s*(\d)", r"\1,\2", text)

    # currency spacing "$ 2.9" -> "$2.9"
    text = re.sub(r"([$€£])\s+(\d)", r"\1\2", text)

    return text

def normalize_brackets(text: str) -> str:
    # "( text )" -> "(text)"
    text = re.sub(r"\(\s+", "(", text)
    text = re.sub(r"\s+\)", ")", text)

    text = re.sub(r"\[\s+", "[", text)
    text = re.sub(r"\s+\]", "]", text)

    text = re.sub(r"\{\s+", "{", text)
    text = re.sub(r"\s+\}", "}", text)

    return text

def has_verb(text):
    doc = nlp(text)
    return any(token.pos_ == "VERB" for token in doc)

def split_sentences(text):
    text = normalize_punctuation(text)
    text = remove_structural_lines(text)
    text = normalize_numbers(text)
    text = normalize_brackets(text)

    doc = nlp(text)
    sentences = [s.text.strip() for s in doc.sents]

    def sentence_filter(sentence):
        # min word count
        if len(sentence.split()) < 4:
            return False
        
        # contain actual words
        if re.match(r"^[\d\s.,-]+$", sentence):
            return False
        
        # requires verb
        if not has_verb(sentence):
            return False
        
        return True

    return list(filter(sentence_filter, sentences))

def argtopk(a, k):
    idx = np.argpartition(a, -k)[-k:]
    return idx[np.argsort(a[idx])[::-1]]

def get_sentence_clusters(sentences, embeddings, top_k):
    # Compute pairwise cosine similarity
    similarity_matrix = cosine_similarity(embeddings)
    sentence_count = len(sentences)
    
    # Exclude self-similarity by setting diagonal to -1
    # Also exclude upper triangle to avoid duplicate pairs
    similarity_matrix[np.tril_indices(sentence_count)] = -1.0

    clusters = [ ]
    for i in range(sentence_count):
        # Select top-k most similar sentences that are above the similarity threshold
        neighbours = [ int(n) for n in argtopk(similarity_matrix[i], top_k) if similarity_matrix[i][n] != -1.0 ]

        # Alternatively, select all sentences above a certain similarity threshold
        #neighbours = [ int(n) for n in range(sentence_count) if similarity_matrix[n][i] > 0.7 ]

        if len(neighbours) > 0:
            clusters.append([ i, *neighbours ])

    return clusters

def predict(premise, hypothesis):
    input = nli_tokenizer(premise, hypothesis, truncation=True, return_tensors="pt").to(device)

    output = nli_model(input["input_ids"])
    prediction = torch.softmax(output["logits"][0], -1).tolist()
    label_names = ["entailment", "neutral", "contradiction"]

    return {name: float(pred) for pred, name in zip(prediction, label_names)}
    
def stringify_claims(claim_indices, claims, prefixes = None):
    sorted_claims = sorted(claim_indices)

    claim_strings = [ ]
    for claim_idx in sorted_claims:
        if prefixes is not None:
            order = sorted_claims.index(claim_idx)

            # Always use last prefix for last claim, otherwise assign prefixes in order of claim index
            if (order == len(claim_indices) - 1):
                prefix = CLAIM_PREFIXES[len(CLAIM_PREFIXES) - 1]
            else:
                prefix = CLAIM_PREFIXES[min(order, len(CLAIM_PREFIXES) - 2)]

            claim_strings.append(f'{prefix}: {claims[claim_idx]}')
        else:
            claim_strings.append(claims[claim_idx])

    return claim_strings

def detect_cluster(cluster, verbose, one_vs_rest=True, hypothesis_prefix=""):
    p_contra = 0

    hypothesis_range = list(range(len(cluster))) if one_vs_rest else [ len(cluster) - 1 ]

    for i in hypothesis_range:
        premise_claims = [ claim for j, claim in enumerate(cluster) if i != j ]
        premise = ' '.join(premise_claims)
        hypothesis = f'{hypothesis_prefix}{cluster[i]}'
    
        prediction = predict(premise, hypothesis)
        p_contra += prediction['contradiction']

        if verbose:
            print(f'Comparing:')
            
            print(f'  {premise}')
            print(f'  {hypothesis}')

            print()

            print(f'  Prediction: {prediction["contradiction"]:.4f}')
            print()

    p_contra = p_contra / len(hypothesis_range)
    return p_contra

def detect(id, document, mode, verbose):
    if mode == 'claims':
        file_name = os.path.join(CLAIMS_INPUT_DIR, f'{id}.txt')
        if os.path.isfile(file_name):
            with open(file_name, 'r', encoding='utf-8') as file:
                claims = [ claim.strip() for claim in file.readlines() if claim.strip() ]
        else:
            # claims = extract_claims(
            #     text=document,
            #     model_name=CLAIM_MODEL_NAME,
            #     backend=CLAIM_MODEL_BACKEND,
            #     temperature=CLAIM_MODEL_TEMPERATURE,
            #     max_new_tokens=CLAIM_MODEL_MAX_NEW_TOKENS,
            #     verbose=CLAIM_MODEL_VERBOSE,
            # )

            # Skip non-cached examples for now
            return None, None, None

    elif mode == 'sentences':
        claims = split_sentences(document)
    else:
        raise "Invalid mode"

    if verbose:
        print()
        print(f'Extracted {len(claims)} claims:')
        print('\n'.join(claims))

    if len(claims) <= 1:
        return None, None, None
    
    embeddings = embed_model.encode(claims)

    single_claims = [ [ i ] for i in range(len(claims)) ]
    clustered_claims = get_sentence_clusters(claims, embeddings, 2)
    clusters = [ *single_claims, *clustered_claims ]

    p_contra_max = 0.0
    evidence = []
    evidence_max = None

    if verbose:
        print()
        print(clustered_claims)
        print()

    for cluster in clusters:
        cluster_strings = stringify_claims(cluster, claims)

        if len(cluster_strings) > 1:
            p_contra = detect_cluster(cluster_strings, verbose, one_vs_rest=False)

            if p_contra > 0.50:
                evidence.append((cluster_strings, p_contra))

                # To reduce false positives, we retest positives with claim prefixes
                #cluster_strings = stringify_claims(cluster, claims, prefixes=CLAIM_PREFIXES)
                p_contra_prefixed = detect_cluster(cluster_strings, verbose, one_vs_rest=False, hypothesis_prefix="But later, ")

                p_contra = (p_contra + p_contra_prefixed) / 2
        else:
            prediction = predict(cluster_strings[0], "")
            p_contra = prediction['contradiction']

            if verbose:
                print(f'Testing:')
                print(f'  {cluster_strings[0]}')
                print(f'  Prediction: {p_contra:.4f}')
                print()

        if p_contra > p_contra_max:
            evidence_max = (cluster_strings, p_contra)
            p_contra_max = p_contra

        if p_contra > 0.50:
            evidence.append((cluster_strings, p_contra))

    return p_contra_max, evidence_max, evidence

In [9]:
# NLI test

contra = "Holmes grows angry when Watson touches items explaining that he doesn't mind his things being touched"

premise = "Miss Delmar is the aunt of the earl de Versely. Captain Delmar frequently visits Miss Delmar at Madeline Hall."  
hypothesis = "Also: Miss Delmar is the aunt of Captain Delmar."

premise = "Ben Keene is ordered back with his detachment. Percival Keene is taken as a prisoner."
hypothesis = "But later, Percival Keene is welcomed back by Captain Delmar."

# premise = "Percival Keene is captured by French troops on the road. Percival Keene is sentenced to execution for spying."
# hypothesis = "But later, Percival Keene and Bob Cross escape during a skirmish."

print(predict(premise, hypothesis))

{'entailment': 0.00017762184143066406, 'neutral': 0.9931640625, 'contradiction': 0.0064544677734375}


In [19]:
# Single evaluation test

pos = False				# Test example from positive or negative set
example_id = None	# Specific example ID to test, or None for random

# 3489738232_6
# story_train_3585
# wiki_train_29443

if example_id is None:
	example_ids = list((dataset['pos' if pos else 'neg']).keys())
	example_id = example_ids[random.randint(0, len(example_ids) - 1)]

example = all_examples[example_id]

print(f'Detecting contradictions in example {example["unique id"]}')
print()

print(example['text'])
print()

if 'evidence' in example:
	print(f'Evidence: {example["evidence"]}')
	print(f'Ref sentences: {example.get("ref sentences", [])}')

print()

p_contra, evidence_max, all_evidence = detect(example['unique id'], example['text'], DETECT_MODE, True)
p_contra, evidence_max, all_evidence

Detecting contradictions in example story_train_2551

 The novel is told from the perspective of an unnamed medical professor. He tells the story of a student named Bernard Langdon, who has to take some time away from his studies to earn money as a teacher. Langdon spends a short time teaching at a school in the village of Pigwacket Centre where he earns respect after taking on the school bully, Abner Briggs. After only a month, however, Langdon leaves to work at the Apollinean Female Institute in the town of Rockland. The owner of the institute is the profit-focused Silas Peckham and the schoolmistress is Miss Helen Darley, who is literally working herself to death. One of his students is the 17-year-old Elsie Venner, who purposely sits apart from the other students. She is known for being strange and quick to anger. She is only close to her father Dudley Venner, who she calls by his first name, and her governess, Old Sophy. She also has a friendship with the town physician Dr. Kittre

(0.4912707010904948,
 (["Bernard Langdon shoots his pistol and kills Dick Venner's horse.",
   'Elsie Venner admits her interest in Bernard Langdon.',
   'Bernard Langdon admits he is concerned about Elsie Venner as a friend.'],
  0.4912707010904948),
 [(['Bernard Langdon climbs up several precipitous rock formations.',
    'Bernard Langdon is overtaken by a rattlesnake poised to strike.',
    'Bernard Langdon is intrigued.'],
   0.5219370524088541),
  (['Bernard Langdon is overtaken by a rattlesnake poised to strike.',
    'Bernard Langdon cages a couple of snakes.',
    'Bernard Langdon is injured.'],
   0.6657540003458658),
  (['Bernard Langdon cages a couple of snakes.',
    "Bernard Langdon shoots his pistol and kills Dick Venner's horse.",
    'Bernard Langdon is injured.'],
   0.6646194458007812),
  (["Dick Venner worries Elsie's father might marry Miss Darley.",
    'During her illness, Elsie Venner calls for Miss Darley to attend to her.',
    'Elsie Venner softens enough to t

In [23]:
# Evaluation loop

experiment_id = 'cluster_claims_nli_r4'
output_file = f'./data/results.{experiment_id}.json'
resume = False

# Allow resuming from previous results
if resume and os.path.exists(output_file):
	results = json.load(open(output_file, "r"))
	
	# Runs store lists instead of dicts, convert them
	if type(results) is list:
		results = { x['unique_id']: x for x in results }
else:
	results = { }

def write_output():
	# Write output file
	with open(output_file, "w") as f:
		json.dump(list(results.values()), f, indent=2)


# Create flat list of examples, shuffle to improve live stats accuracy
test_set = list(all_examples.items())
random.shuffle(test_set)

# Enumerate to get indices, convert to list to get progress indication
test_set = list(enumerate(test_set))

for i, (unique_id, example) in tqdm.tqdm(test_set):
	unique_id = example["unique id"]
	text = example["text"]
	y_true = example["label"]        # 0 or 1
	doc_type = example["doc_type"]   # story/wiki/news

	# Skip tests already completed previous run
	if unique_id in results:
		continue

	p_pred, evidence_max, all_evidence = detect(unique_id, text, DETECT_MODE, False)

	# Some examples fail due to claim extraction returning 0 or 1 claims
	if p_pred is None:
		print(f'Warning! Detection failed for {unique_id}')
		continue

	results[unique_id] = {
		"unique_id": unique_id,
		"p_pred": p_pred,
		"y_true": y_true,
		"doc_type": doc_type
	}

	# Ocassionally write output, to get live stats and in case of crash
	if i % 10 == 0:
		write_output()

write_output()

100%|██████████| 891/891 [1:45:24<00:00,  7.10s/it]  
